# MedViLL — Modernized
**Medical Vision-Language Learning** | Retrieval · Classification · VQA · Report Generation

---

### Credits & Datasets
- **Original paper**: Moon et al., *"Multi-modal Understanding and Generation for Medical Images and Text via Vision-Language Pre-Training"*, IEEE JBHI 2022. [GitHub](https://github.com/SuperSupermoon/MedViLL)
- **OpenI (Indiana University CXR)** *(used for classification, retrieval, generation)*: Demner-Fushman et al., *"Preparing a collection of radiology examinations for distribution and retrieval"*, JAMIA 2016. DOI: [10.1093/jamia/ocv080](https://doi.org/10.1093/jamia/ocv080)
- **VQA-RAD** *(used for VQA only)*: Lau et al., *"A dataset of clinically generated visual questions and answers about radiology images"*, Scientific Data 2018. DOI: [10.1038/sdata.2018.251](https://doi.org/10.1038/sdata.2018.251)

> **Why OpenI instead of MIMIC-CXR?** MIMIC-CXR requires a credentialed PhysioNet account and cannot be accessed programmatically from Colab without manual download. OpenI is freely downloadable from the NLM without registration.

---
**Runtime**: GPU → Runtime → Change runtime type → T4 GPU  
**Repo**: https://github.com/schizocatto/medvill_fix

## 0 — Environment check

In [ ]:
import torch, platform, subprocess

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"GPU     : {gpu}")
print(f"RAM     : {subprocess.getoutput('free -h | grep Mem').split()[1]}"
      if platform.system() == "Linux" else "")

## 1 — Clone repo & install dependencies

In [ ]:
import os

REPO_URL  = "https://github.com/schizocatto/medvill_fix.git"
REPO_DIR  = "/content/medvill_fix"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
%%capture install_log
# Colab already has torch; we only need the extra deps.
# --quiet keeps the cell output clean; check install_log if anything fails.
!pip install -q \
    transformers>=4.40.0 \
    accelerate>=0.29.0 \
    einops>=0.7.0 \
    sacrebleu>=2.3.1 \
    rouge-score>=0.1.2 \
    omegaconf>=2.3.0 \
    evaluate>=0.4.1 \
    fuzzywuzzy>=0.18.0 \
    python-Levenshtein>=0.25.0

print("✓ Dependencies installed")

In [ ]:
import sys, importlib

# Add repo root to path so `import medvill` works
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Smoke-test core imports
import medvill
from medvill.config import MedViLLConfig, ModelConfig, ImageConfig, TrainingConfig
from medvill.models import (
    MedViLL,
    MedViLLForClassification,
    MedViLLForRetrieval,
    MedViLLForVQA,
    MedViLLForGeneration,
)
from medvill.metrics import (
    compute_bleu4, RunningPerplexity,
    compute_retrieval_metrics,
    compute_classification_metrics,
)
from medvill.utils import set_seed, get_logger

print(f"✓ medvill {medvill.__version__} imported successfully")
print(f"  device: {'cuda' if __import__('torch').cuda.is_available() else 'cpu'}")

## 2 — Mount Google Drive & configure paths

Upload the OpenI archives to your Drive before running this section.  
Download links (no login required):
- Images : https://openi.nlm.nih.gov/imgs/collections/NLMCXR_png.tgz  (~1 GB)
- Reports: https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz  (~4 MB)

Expected Drive layout after you upload:
```
MyDrive/MedViLL/
  raw/
    NLMCXR_png.tgz
    NLMCXR_reports.tgz
  vqa_rad/            ← VQA-RAD directory (trainset.json, testset.json, images/, cache/)
  checkpoints/        ← optional pre-training checkpoint
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ── Edit only this block ──────────────────────────────────────────────────────
DRIVE_ROOT    = "/content/drive/MyDrive/MedViLL"
PRETRAIN_CKPT = f"{DRIVE_ROOT}/checkpoints/pretrain_best.pt"  # set None to skip

# Raw OpenI archives (uploaded by you to Drive)
OPENI_PNG_TGZ     = f"{DRIVE_ROOT}/raw/NLMCXR_png.tgz"
OPENI_REPORTS_TGZ = f"{DRIVE_ROOT}/raw/NLMCXR_reports.tgz"

# Extracted / processed paths (auto-populated in next cells)
OPENI_IMG_DIR  = "/content/openi_images"        # extracted PNGs land here
OPENI_XML_DIR  = "/content/openi_reports"        # extracted XMLs land here
OPENI_DATA_DIR = "/content/openi_jsonl"          # our JSONL splits land here

# VQA-RAD (still from Drive — not replaced)
VQA_DIR        = f"{DRIVE_ROOT}/vqa_rad"

OUTPUT_DIR     = f"{DRIVE_ROOT}/outputs"
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Paths configured.")

### 2a — Extract & parse OpenI → JSONL splits

Parses the NLM XML reports, pairs them with their PNG images, and writes
train / val / test JSONL files. Splits are deterministic (seed 42): 70 / 15 / 15.

In [ ]:
import os, tarfile, json, random, glob
import xml.etree.ElementTree as ET
from pathlib import Path

random.seed(42)

# ── 1. Extract archives ───────────────────────────────────────────────────────
def extract_if_needed(tgz_path, dest_dir):
    if not os.path.exists(dest_dir) or not os.listdir(dest_dir):
        os.makedirs(dest_dir, exist_ok=True)
        print(f"Extracting {Path(tgz_path).name} ...")
        with tarfile.open(tgz_path, "r:gz") as tar:
            tar.extractall(dest_dir)
        print(f"  → {dest_dir}")
    else:
        print(f"  Already extracted: {dest_dir}")

extract_if_needed(OPENI_REPORTS_TGZ, OPENI_XML_DIR)
extract_if_needed(OPENI_PNG_TGZ,     OPENI_IMG_DIR)

# ── 2. Build image index (stem → absolute path) ───────────────────────────────
img_index = {}
for p in Path(OPENI_IMG_DIR).rglob("*.png"):
    img_index[p.stem] = str(p)
print(f"Indexed {len(img_index):,} PNG images")

# ── 3. Parse XML reports ──────────────────────────────────────────────────────
def parse_report(xml_path):
    """Return dict with keys: uid, img_stems, text, labels, impression, findings."""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    uid = (root.findtext(".//uId") or Path(xml_path).stem).strip()

    # Text fields
    findings  = root.findtext(".//AbstractText[@Label='FINDINGS']")  or ""
    impression = root.findtext(".//AbstractText[@Label='IMPRESSION']") or ""
    # Combine; at least one is usually present
    text = " ".join(filter(None, [findings.strip(), impression.strip()]))

    # MeSH labels (major = pathology labels)
    major_labels = [m.text.strip() for m in root.findall(".//MeSH/major") if m.text]

    # Associated image file stems  (attribute "id" on <parentImage> elements)
    img_stems = [el.get("id","").strip()
                 for el in root.findall(".//parentImage") if el.get("id")]

    return {
        "uid": uid,
        "img_stems": img_stems,
        "text": text,
        "label": ", ".join(major_labels),
        "impression": impression.strip(),
        "findings": findings.strip(),
    }

xml_files = list(Path(OPENI_XML_DIR).rglob("*.xml"))
print(f"Found {len(xml_files):,} XML report files")

records = []
for xf in xml_files:
    r = parse_report(xf)
    if not r["text"]:        # skip reports with no text
        continue
    # Match report to its front-view image (prefer frontal: no "_lat" in stem)
    matched_img = None
    for stem in r["img_stems"]:
        if stem in img_index and "_lat" not in stem.lower():
            matched_img = img_index[stem]
            break
    if matched_img is None:   # fallback: accept any view
        for stem in r["img_stems"]:
            if stem in img_index:
                matched_img = img_index[stem]
                break
    if matched_img is None:   # no image found → skip
        continue
    records.append({
        "uid":       r["uid"],
        "img":       matched_img,
        "text":      r["text"],
        "label":     r["label"],
        "impression": r["impression"],
        "findings":  r["findings"],
    })

print(f"Usable (image + text) records: {len(records):,}")

# ── 4. Split 70 / 15 / 15 ─────────────────────────────────────────────────────
random.shuffle(records)
n = len(records)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
splits  = {
    "train": records[:n_train],
    "val":   records[n_train:n_train + n_val],
    "test":  records[n_train + n_val:],
}

# ── 5. Write JSONL files ──────────────────────────────────────────────────────
os.makedirs(OPENI_DATA_DIR, exist_ok=True)
for split, recs in splits.items():
    out = os.path.join(OPENI_DATA_DIR, f"{split}.jsonl")
    with open(out, "w") as f:
        for rec in recs:
            f.write(json.dumps(rec) + "\n")
    print(f"  {split:5s}: {len(recs):,} records → {out}")

# ── 6. Convenience variables used by all downstream cells ────────────────────
CLS_TRAIN = GEN_TRAIN = RET_TRAIN = os.path.join(OPENI_DATA_DIR, "train.jsonl")
CLS_VAL   = GEN_VAL   = RET_VAL   = os.path.join(OPENI_DATA_DIR, "val.jsonl")
CLS_TEST  = GEN_TEST  = RET_TEST  = os.path.join(OPENI_DATA_DIR, "test.jsonl")
print("\n✓ JSONL splits ready. Variables CLS_*/RET_*/GEN_* all point to OpenI splits.")

## 3 — Exploratory Data Analysis (OpenI)

Understand the dataset before training: corpus statistics, label distribution, report-length distribution, image statistics, and sample visualisation.

In [ ]:
import json
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path

# Load full dataset (all splits)
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train_recs = load_jsonl(CLS_TRAIN)
val_recs   = load_jsonl(CLS_VAL)
test_recs  = load_jsonl(CLS_TEST)
all_recs   = train_recs + val_recs + test_recs

# ── Corpus overview ───────────────────────────────────────────────────────────
print("=" * 52)
print(f"{'OpenI Dataset Overview':^52}")
print("=" * 52)
print(f"  Total records        : {len(all_recs):>6,}")
print(f"  Train                : {len(train_recs):>6,}  ({100*len(train_recs)/len(all_recs):.0f}%)")
print(f"  Val                  : {len(val_recs):>6,}  ({100*len(val_recs)/len(all_recs):.0f}%)")
print(f"  Test                 : {len(test_recs):>6,}  ({100*len(test_recs)/len(all_recs):.0f}%)")

# Token / word counts
word_counts = [len(r["text"].split()) for r in all_recs]
char_counts = [len(r["text"]) for r in all_recs]
print(f"\n  Report word count    : mean={np.mean(word_counts):.1f}  "
      f"median={np.median(word_counts):.0f}  max={max(word_counts)}")
print(f"  Report char count    : mean={np.mean(char_counts):.1f}  "
      f"max={max(char_counts)}")
print(f"  Empty reports        : {sum(1 for r in all_recs if not r['text'].strip())}")

# Labels
label_counts_all = Counter()
records_no_label = 0
for r in all_recs:
    lbls = [l.strip() for l in r.get("label","").split(",") if l.strip()]
    if lbls:
        label_counts_all.update(lbls)
    else:
        records_no_label += 1

print(f"\n  Unique MeSH labels   : {len(label_counts_all):>6,}")
print(f"  Records w/ no label  : {records_no_label:>6,}  "
      f"({100*records_no_label/len(all_recs):.1f}%)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Figure 1: Label distribution (top-25) ────────────────────────────────────
top_n = 25
top_labels = label_counts_all.most_common(top_n)
lbl_names  = [l for l, _ in top_labels]
lbl_vals   = [c for _, c in top_labels]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("OpenI Dataset — EDA", fontsize=15, fontweight="bold")

ax = axes[0]
bars = ax.barh(lbl_names[::-1], lbl_vals[::-1], color="steelblue", edgecolor="white")
ax.set_xlabel("Count (across all splits)")
ax.set_title(f"Top-{top_n} MeSH Pathology Labels")
for bar, val in zip(bars, lbl_vals[::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            str(val), va="center", fontsize=8)
ax.set_xlim(0, max(lbl_vals) * 1.15)

# ── Figure 2: Report word-length distribution ─────────────────────────────────
ax2 = axes[1]
ax2.hist(word_counts, bins=50, color="darkorange", edgecolor="white", alpha=0.85)
ax2.axvline(np.mean(word_counts),   color="red",   linestyle="--", lw=1.5,
            label=f"mean={np.mean(word_counts):.0f}")
ax2.axvline(np.median(word_counts), color="green", linestyle="--", lw=1.5,
            label=f"median={np.median(word_counts):.0f}")
ax2.set_xlabel("Words per report")
ax2.set_ylabel("Number of reports")
ax2.set_title("Report Length Distribution")
ax2.legend()

plt.tight_layout()
plt.savefig("/content/eda_labels_length.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
from PIL import Image
import numpy as np

# ── Image statistics ──────────────────────────────────────────────────────────
sample_paths = [r["img"] for r in all_recs[:500]]   # sample 500 for speed
widths, heights, aspects = [], [], []
for p in sample_paths:
    try:
        w, h = Image.open(p).size
        widths.append(w); heights.append(h); aspects.append(w/h)
    except Exception:
        pass

print(f"Image sample size : {len(widths)}")
print(f"Width  — mean: {np.mean(widths):.0f}  min: {min(widths)}  max: {max(widths)}")
print(f"Height — mean: {np.mean(heights):.0f}  min: {min(heights)}  max: {max(heights)}")
print(f"Aspect — mean: {np.mean(aspects):.2f}  (1.0 = square)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(widths,  bins=30, color="teal",   edgecolor="white", alpha=0.8)
axes[0].set_title("Image Width Distribution (sample)")
axes[0].set_xlabel("Pixels")
axes[1].hist(heights, bins=30, color="purple", edgecolor="white", alpha=0.8)
axes[1].set_title("Image Height Distribution (sample)")
axes[1].set_xlabel("Pixels")
plt.tight_layout()
plt.savefig("/content/eda_image_sizes.png", dpi=120, bbox_inches="tight")
plt.show()

# ── Labels per record ─────────────────────────────────────────────────────────
labels_per_rec = [len([l for l in r.get("label","").split(",") if l.strip()])
                  for r in all_recs]
print(f"\nLabels per record — mean: {np.mean(labels_per_rec):.2f}  "
      f"median: {np.median(labels_per_rec):.0f}  "
      f"max: {max(labels_per_rec)}")
print(f"Records with 0 labels : {labels_per_rec.count(0):,}")

In [ ]:
import random, textwrap
from PIL import Image
import matplotlib.pyplot as plt

# ── Sample image + report grid (4 random training records) ───────────────────
random.seed(7)
samples = random.sample(train_recs, k=min(4, len(train_recs)))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle("OpenI — Sample Images and Reports", fontsize=13, fontweight="bold")

for i, rec in enumerate(samples):
    # Image column
    ax_img = axes[0][i]
    try:
        img = Image.open(rec["img"]).convert("L")
        ax_img.imshow(img, cmap="gray")
    except Exception as e:
        ax_img.text(0.5, 0.5, f"[load error]\n{e}", ha="center", va="center",
                    transform=ax_img.transAxes, fontsize=7)
    ax_img.axis("off")
    lbls = rec.get("label", "—") or "—"
    ax_img.set_title(f"Labels:\n{textwrap.fill(lbls, 30)}", fontsize=7)

    # Report column
    ax_txt = axes[1][i]
    ax_txt.axis("off")
    wrapped = textwrap.fill(rec["text"][:500], width=42)
    if len(rec["text"]) > 500:
        wrapped += "\n[...]"
    ax_txt.text(0.02, 0.98, wrapped, transform=ax_txt.transAxes,
                fontsize=7, va="top", family="monospace",
                bbox=dict(boxstyle="round,pad=0.4", fc="#f5f5f5", ec="#cccccc"))

plt.tight_layout()
plt.savefig("/content/eda_sample_images.png", dpi=120, bbox_inches="tight")
plt.show()

## 4 — Metrics reference

All formulas used in this notebook. $Q$ = set of queries, $N$ = number of tokens, $C$ = number of classes.

---

### 4.1 Retrieval — Recall@K and MRR

$$
\text{Recall@K} = \frac{1}{|Q|} \sum_{q \in Q} \mathbf{1}\!\left[\text{rank}(q) \leq K\right]
$$

$$
\text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \frac{1}{\text{rank}(q)}
$$

where $\text{rank}(q)$ is the position of the correct item in the score-sorted list (1-indexed). I2T = image query → text gallery; T2I = text query → image gallery. Scores come from the ITM head: $s = P(\text{aligned} \mid \mathbf{img}, \mathbf{txt})$.

---

### 4.2 Classification — AUC-ROC and macro-F1

$$
\text{AUC} = \int_0^1 \text{TPR}\!\left(\text{FPR}^{-1}(t)\right) dt
\qquad \in [0,1],\quad 0.5 = \text{random}
$$

Per-class precision and recall:
$$
P_c = \frac{TP_c}{TP_c + FP_c}, \quad R_c = \frac{TP_c}{TP_c + FN_c}, \quad F1_c = \frac{2 P_c R_c}{P_c + R_c}
$$

Macro averages (equal weight per class regardless of support):
$$
\overline{\text{AUC}} = \frac{1}{C}\sum_{c=1}^C \text{AUC}_c, \qquad \overline{F1} = \frac{1}{C}\sum_{c=1}^C F1_c
$$

---

### 4.3 Generation — BLEU-4 and Perplexity

**BLEU-4** (Papineni et al., 2002):
$$
\text{BLEU-4} = BP \cdot \exp\!\left(\frac{1}{4}\sum_{n=1}^{4} \log p_n\right)
$$

where the modified $n$-gram precision is:
$$
p_n = \frac{\displaystyle\sum_{\text{ngram} \in \hat{y}} \min\!\left(\text{count}(\text{ngram},\hat{y}),\;\text{count}(\text{ngram},y)\right)}{\displaystyle\sum_{\text{ngram} \in \hat{y}} \text{count}(\text{ngram},\hat{y})}
$$

and the brevity penalty prevents rewarding trivially short outputs:
$$
BP = \begin{cases} 1 & \text{if } c > r \\ e^{1 - r/c} & \text{if } c \leq r \end{cases}
$$

with $c$ = hypothesis length, $r$ = reference length.

**Perplexity** (lower is better; $\infty$ at random initialisation):
$$
\text{PPL} = \exp\!\left(-\frac{1}{N}\sum_{i=1}^{N} \log P_\theta(w_i \mid w_{<i}, \mathbf{img})\right)
$$

In code, this is computed from the teacher-forced cross-entropy loss $\mathcal{L}$ as $\text{PPL} = e^{\mathcal{L}}$.

---

### 4.4 VQA — Accuracy and BLEU-4

$$
\text{Accuracy} = \frac{1}{|D|}\sum_{(q,a,\hat{a}) \in D} \mathbf{1}[\hat{a} = a]
$$

BLEU-4 is also computed treating each predicted answer string as a hypothesis and the ground-truth answer as the reference (single-sentence BLEU), useful for open-ended questions where multiple valid phrasings exist.

## 3 — Shared config & tokenizer

One config object is shared across all experiments; override per-task fields inline.

In [ ]:
import torch
from omegaconf import OmegaConf
from transformers import AutoTokenizer
from medvill.config import MedViLLConfig, ModelConfig, ImageConfig, DataConfig, TrainingConfig
from medvill.utils import set_seed

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ── Base config (tweak any field here) ───────────────────────────────────────
cfg = MedViLLConfig(
    model=ModelConfig(bert_model="bert-base-uncased"),
    image=ImageConfig(
        encoder_type="resnet50",
        img_size=512,
        num_image_embeds=180,
        img_hidden_sz=2048,
    ),
    data=DataConfig(
        train_path="PLACEHOLDER",   # overridden per task
        seq_len=128,
        num_workers=2,              # 2 is safe on Colab
    ),
    training=TrainingConfig(
        output_dir=OUTPUT_DIR,
        epochs=10,
        batch_size=16,              # reduce to 8 if OOM
        lr=5e-5,
        fp16=True,
        seed=42,
        gradient_accumulation_steps=2,
    ),
)

TOKENIZER = AutoTokenizer.from_pretrained(cfg.model.bert_model)
print(f"Tokenizer vocab size: {TOKENIZER.vocab_size}")

## 5 — Experiment A: Diagnosis Classification (OpenI)

Multi-label classification over OpenI MeSH pathology labels.  
**Metrics** (§4.2): macro AUC-ROC $\overline{\text{AUC}}$, macro F1 $\overline{F1}$, per-label AUC.

In [ ]:
import json
from torch.utils.data import DataLoader
from medvill.data import ClassificationDataset
from medvill.models import MedViLLForClassification
from medvill.tasks import ClassificationTrainer
from medvill.utils import load_pretrained_encoder

# ── Build label map from training file ───────────────────────────────────────
def build_label_map(jsonl_path):
    labels = set()
    with open(jsonl_path) as f:
        for line in f:
            raw = json.loads(line).get("label", "")
            if isinstance(raw, str):
                labels.update(l.strip() for l in raw.split(",") if l.strip())
    return {l: i for i, l in enumerate(sorted(labels))}

label2idx   = build_label_map(CLS_TRAIN)
label_names = [k for k, v in sorted(label2idx.items(), key=lambda x: x[1])]
print(f"Labels ({len(label2idx)}): {label_names}")

# ── Datasets & loaders ───────────────────────────────────────────────────────
cls_cfg = cfg   # re-use shared config
cls_cfg.data.train_path = CLS_TRAIN
cls_cfg.data.val_path   = CLS_VAL

train_ds = ClassificationDataset(CLS_TRAIN, TOKENIZER, label2idx,
                                  seq_len=cfg.data.seq_len, img_size=cfg.image.img_size,
                                  train=True)
val_ds   = ClassificationDataset(CLS_VAL,   TOKENIZER, label2idx,
                                  seq_len=cfg.data.seq_len, img_size=cfg.image.img_size,
                                  train=False)

train_loader = DataLoader(train_ds, batch_size=cfg.training.batch_size,
                          shuffle=True,  num_workers=cfg.data.num_workers, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.training.batch_size,
                          shuffle=False, num_workers=cfg.data.num_workers)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

# ── Model ─────────────────────────────────────────────────────────────────────
cls_model = MedViLLForClassification(cfg.model, cfg.image,
                                      num_labels=len(label2idx), multilabel=True)
if PRETRAIN_CKPT and os.path.exists(PRETRAIN_CKPT):
    missing, unexpected = load_pretrained_encoder(cls_model, PRETRAIN_CKPT)
    print(f"Loaded pretrained encoder | missing={len(missing)} unexpected={len(unexpected)}")

# ── Train ─────────────────────────────────────────────────────────────────────
cls_trainer = ClassificationTrainer(cls_model, train_loader, val_loader,
                                     cfg, DEVICE, label_names)
cls_trainer.train()

## 6 — Experiment B: Retrieval I2T / T2I (OpenI)

Cross-encoder retrieval: ITM head scores every (image, report) pair; all pairs ranked by $P(\text{aligned})$.  
**Metrics** (§4.1): Recall@1/5/10, MRR, median rank — both I2T and T2I directions.

In [ ]:
from medvill.data import RetrievalDataset, RetrievalEvalDataset
from medvill.models import MedViLLForRetrieval
from medvill.tasks import RetrievalTrainer

ret_train_ds = RetrievalDataset(RET_TRAIN, TOKENIZER,
                                 seq_len=cfg.data.seq_len, img_size=cfg.image.img_size,
                                 train=True)
ret_eval_ds  = RetrievalEvalDataset(RET_VAL, TOKENIZER,
                                     seq_len=cfg.data.seq_len, img_size=cfg.image.img_size)

ret_train_loader = DataLoader(ret_train_ds, batch_size=cfg.training.batch_size,
                               shuffle=True, num_workers=cfg.data.num_workers, pin_memory=True)

print(f"Train: {len(ret_train_ds)} | Eval: {len(ret_eval_ds)}")

ret_model = MedViLLForRetrieval(cfg.model, cfg.image)
if PRETRAIN_CKPT and os.path.exists(PRETRAIN_CKPT):
    load_pretrained_encoder(ret_model, PRETRAIN_CKPT)
    print("Loaded pretrained encoder")

ret_trainer = RetrievalTrainer(ret_model, ret_train_loader, ret_eval_ds, cfg, DEVICE)
ret_trainer.train()

## 7 — Experiment C: VQA on VQA-RAD

Visual Question Answering on radiology images (VQA-RAD dataset).  
Set `VQA_ORGAN` to `"chest"` / `"head"` / `"abd"` or `None` for all organs.  
**Metrics** (§4.4): Accuracy (exact match), BLEU-4 on answer strings.

In [ ]:
from medvill.data import VQARadDataset
from medvill.models import MedViLLForVQA
from medvill.tasks import VQATrainer
from medvill.metrics import compute_bleu4

VQA_ORGAN = None   # "chest" | "head" | "abd" | None

vqa_train_ds = VQARadDataset(VQA_DIR, TOKENIZER, split="train", organ=VQA_ORGAN,
                              seq_len=cfg.data.seq_len, img_size=cfg.image.img_size, train=True)
vqa_test_ds  = VQARadDataset(VQA_DIR, TOKENIZER, split="test",  organ=VQA_ORGAN,
                              seq_len=cfg.data.seq_len, img_size=cfg.image.img_size, train=False)

# Share vocab built from train set
vqa_test_ds.ans2label = vqa_train_ds.ans2label
vqa_test_ds.label2ans = vqa_train_ds.label2ans
vqa_test_ds.num_answers = vqa_train_ds.num_answers

print(f"Train: {len(vqa_train_ds)} | Test: {len(vqa_test_ds)}")
print(f"Answer vocab size: {vqa_train_ds.num_answers}")

vqa_train_loader = DataLoader(vqa_train_ds, batch_size=cfg.training.batch_size,
                               shuffle=True,  num_workers=cfg.data.num_workers, pin_memory=True)
vqa_test_loader  = DataLoader(vqa_test_ds,  batch_size=cfg.training.batch_size,
                               shuffle=False, num_workers=cfg.data.num_workers)

vqa_model = MedViLLForVQA(cfg.model, cfg.image, num_answers=vqa_train_ds.num_answers)
if PRETRAIN_CKPT and os.path.exists(PRETRAIN_CKPT):
    load_pretrained_encoder(vqa_model, PRETRAIN_CKPT)
    print("Loaded pretrained encoder")

vqa_trainer = VQATrainer(vqa_model, vqa_train_loader, vqa_test_loader,
                          cfg, DEVICE, label2ans=vqa_train_ds.label2ans)
vqa_trainer.train()

# ── Post-training metrics ─────────────────────────────────────────────────────
print("\n── Test-set metrics ──")
pred_texts, gt_texts = vqa_trainer.predict(vqa_test_loader)
bleu = compute_bleu4(pred_texts, gt_texts)
print(f"BLEU-4 : {bleu['bleu4']:.2f}")
print(f"BLEU-1 : {bleu['bleu1']:.2f}")
print(f"BLEU-2 : {bleu['bleu2']:.2f}")
print(f"BLEU-3 : {bleu['bleu3']:.2f}")

## 8 — Experiment D: Report Generation (OpenI)

Seq2seq radiology report generation from chest X-ray images.  
**Metrics** (§4.3): BLEU-4 $\in[0,100]$ (higher = better), Perplexity $\in[1,\infty)$ (lower = better).

In [ ]:
from medvill.data import ReportGenerationDataset
from medvill.models import MedViLLForGeneration
from medvill.tasks import GenerationTrainer

gen_train_ds = ReportGenerationDataset(GEN_TRAIN, TOKENIZER,
                                        seq_len=cfg.data.seq_len, img_size=cfg.image.img_size,
                                        train=True)
gen_val_ds   = ReportGenerationDataset(GEN_VAL,   TOKENIZER,
                                        seq_len=cfg.data.seq_len, img_size=cfg.image.img_size,
                                        train=False)

gen_train_loader = DataLoader(gen_train_ds, batch_size=cfg.training.batch_size,
                               shuffle=True,  num_workers=cfg.data.num_workers, pin_memory=True)
gen_val_loader   = DataLoader(gen_val_ds,   batch_size=cfg.training.batch_size,
                               shuffle=False, num_workers=cfg.data.num_workers)

print(f"Train: {len(gen_train_ds)} | Val: {len(gen_val_ds)}")

gen_model = MedViLLForGeneration(cfg.model, cfg.image)
if PRETRAIN_CKPT and os.path.exists(PRETRAIN_CKPT):
    load_pretrained_encoder(gen_model, PRETRAIN_CKPT)
    print("Loaded pretrained encoder")

gen_trainer = GenerationTrainer(gen_model, gen_train_loader, gen_val_loader,
                                 TOKENIZER, cfg, DEVICE)
gen_trainer.train()

## 9 — Standalone evaluation on a saved checkpoint

Evaluate any saved checkpoint without retraining. Set `EVAL_TASK` and `EVAL_CHECKPOINT` below.

In [ ]:
EVAL_TASK       = "vqa"        # classification | retrieval | vqa | generation
EVAL_CHECKPOINT = f"{OUTPUT_DIR}/vqa/vqa_best_epoch009_step0.pt"   # adjust path

import subprocess, sys

cmd = [
    sys.executable, f"{REPO_DIR}/scripts/evaluate.py",
    "--task",       EVAL_TASK,
    "--config",     f"{REPO_DIR}/configs/{EVAL_TASK}.yaml",
    "--checkpoint", EVAL_CHECKPOINT,
]

# Task-specific path args
if EVAL_TASK == "vqa":
    cmd += ["--vqa_dir", VQA_DIR]
elif EVAL_TASK == "classification":
    cmd += ["--data_path", CLS_TEST]
elif EVAL_TASK == "retrieval":
    cmd += ["--data_path", RET_VAL]
elif EVAL_TASK == "generation":
    cmd += ["--data_path", GEN_VAL]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])